In [ ]:
import pandas as pd
from datasets import load_dataset
import psycopg
from pgvector.psycopg import register_vector
from sentence_transformers import SentenceTransformer
import os
from dotenv import load_dotenv
import os
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_correctness,
    answer_similarity,
    context_entity_recall
)
from ragas.llms import LangchainLLMWrapper
load_dotenv()

True

In [2]:
df_hybrid       = pd.read_parquet("parquet/multilingual-e5-large/embeddings_docling_hybrid.parquet")
df_hierarchical = pd.read_parquet("parquet/multilingual-e5-large/embeddings_docling_hierarchical.parquet")
df_mdheader     = pd.read_parquet("parquet/multilingual-e5-large/embeddings_langchain_mdheader.parquet")
df_recursive    = pd.read_parquet("parquet/multilingual-e5-large/embeddings_langchain_recursive.parquet")

target_doc_ids = list(
    set(df_hybrid["doc_id"]) & 
    set(df_hierarchical["doc_id"]) & 
    set(df_mdheader["doc_id"]) & 
    set(df_recursive["doc_id"])
)
target_doc_ids

['2412.15239v2',
 '2405.16945v4',
 '2404.18137v2',
 '2401.12638v7',
 '2412.08070v2',
 '2410.05184v2',
 '2409.15621v2',
 '2406.02319v2',
 '2408.13702v3',
 '2412.17354v3',
 '2412.21038v2',
 '2406.01505v2',
 '2408.09838v2',
 '2411.12110v2',
 '2410.14077v2',
 '2401.03305v2',
 '2409.13333v2',
 '2412.11336v3',
 '2406.15888v2',
 '2410.24112v2',
 '2403.08753v2',
 '2403.01421v2',
 '2412.17119v3',
 '2404.17509v2',
 '2409.18681v6',
 '2412.02459v2',
 '2409.04843v2',
 '2412.14566v2',
 '2401.16412v4',
 '2410.23841v2',
 '2411.05803v4',
 '2410.08287v3',
 '2405.20415v3',
 '2412.11338v2',
 '2406.02969v2',
 '2411.02796v2',
 '2411.18086v2',
 '2409.14494v3',
 '2405.08806v2',
 '2403.05412v2',
 '2408.04814v3',
 '2411.15293v2',
 '2408.11878v2',
 '2404.09796v2',
 '2404.17994v2',
 '2407.09711v4',
 '2412.00267v2',
 '2411.03676v2',
 '2408.07618v3',
 '2409.13674v3',
 '2412.11355v2',
 '2403.11010v3',
 '2407.02507v2',
 '2408.11915v2',
 '2409.14616v2',
 '2407.14986v2',
 '2403.06613v3',
 '2402.04429v3',
 '2411.12375v3

In [3]:
ds = load_dataset("G4KMU/vectara_open_ragbench", "Open RAGBench", split="text_tables")
ds

Dataset({
    features: ['id', 'context_id', 'split', 'question', 'answer', 'context', 'tables', 'img_paths', 'text', 'question_type', 'doc_id', 'section_id', 'title', 'authors', 'categories', 'abstract', 'updated', 'published', 'context_length'],
    num_rows: 2062
})

In [4]:
ds_filtered = ds.filter(lambda x: x['doc_id'] in target_doc_ids)

# Kurzer Check
print(f"Ursprüngliche Größe: {len(ds)}")
print(f"Gefilterte Größe: {len(ds_filtered)}")

df_filtered = ds_filtered.to_pandas()
ds_filtered[0]

Filter:   0%|          | 0/2062 [00:00<?, ? examples/s]

Ursprüngliche Größe: 2062
Gefilterte Größe: 532


{'id': '947fbbd3-465b-48ad-bc13-068dd830b215',
 'context_id': '2412.15239v2_6',
 'split': 'text',
 'question': 'How are expectations calculated in the narrative framework?',
 'answer': 'Expectations are calculated as the mean of each feature across all imagined continuations for a given chapter.',
 'context': '# 3.3 Feature Extraction \n\nWith multiple imagined story continuations per chapter, the next question is how to quantify the unstructured story text. We approach this question in two steps: 1) we extract from the text predefined features that have been proposed in the literature to be associated with narrative success and 2) we calculate measures of expectations, uncertainty, and surprise based on those features. Let $i$ represent the focal book, $t$ the focal chapter, and $n$ the imagined story number with $N$ capturing the total number of imagined stories per chapter. Let $z_{i t n}$ denote the extracted features from the text (i.e., $z_{i t n}=f\\left(\\right.$ ImaginedStory 

In [5]:
model = SentenceTransformer("intfloat/multilingual-e5-large")

def retrieve_from_db(question, table_name, top_k=3):
    conn = psycopg.connect("dbname=chunks_evaluation user=admin password=admin host=localhost port=59101")
    register_vector(conn)
    
    query_embedding = model.encode(f"query: {question}")
    
    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT content FROM {table_name}
            ORDER BY embedding <=> %s
            LIMIT %s
        """, (query_embedding, top_k))
        
        results = [row[0] for row in cur.fetchall()]
    conn.close()
    return results

In [ ]:
ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
KEY = os.getenv("AZURE_OPENAI_API_KEY")
API_VERSION = os.getenv("AZURE_API_VERSION")
DEPLOYMENT_CHAT = os.getenv("AZURE_DEPLOYMENT_CHAT")
DEPLOYMENT_EMBED = os.getenv("AZURE_DEPLOYMENT_EMBED")

# --- 1. AZURE CONFIGURATION ---
os.environ["AZURE_OPENAI_API_KEY"] = KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = ENDPOINT
os.environ["OPENAI_API_KEY"] = "dummy-key-for-internal-checks"

azure_model = AzureChatOpenAI(
    azure_deployment=DEPLOYMENT_CHAT,
    openai_api_version=API_VERSION,
)

AZURE_CONFIG = {
    "endpoint": ENDPOINT,
    "api_key": KEY,
    "api_version": API_VERSION,
    "deployment_chat": DEPLOYMENT_CHAT,
    "deployment_embed": DEPLOYMENT_EMBED
}

azure_model = AzureChatOpenAI(
    azure_endpoint=AZURE_CONFIG["endpoint"],
    azure_deployment=AZURE_CONFIG["deployment_chat"],
    openai_api_key=AZURE_CONFIG["api_key"],
    openai_api_version=AZURE_CONFIG["api_version"],
)
ragas_llm = LangchainLLMWrapper(azure_model)

azure_embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_CONFIG["endpoint"],
    azure_deployment=AZURE_CONFIG["deployment_embed"],
    openai_api_key=AZURE_CONFIG["api_key"],
    openai_api_version=AZURE_CONFIG["api_version"],
)
ragas_emb = LangchainEmbeddingsWrapper(azure_embeddings)

# --- 2. SETUP DER EVALUIERUNG ---
# 4 Tabellennamen aus pgvector
tabellen = [
    "chunks_docling_hybrid",
    "chunks_docling_hierarchical",
    # "chunks_langchain_mdheader",
    "chunks_langchain_recursive"
]

SEED = 42
ds_shuffled = ds_filtered.shuffle(seed=SEED)

anzahl_samples = min(200, len(ds_shuffled))

test_queries = [
    {"question": x['question'], "ground_truth": x['answer']} 
    for x in ds_shuffled.select(range(anzahl_samples))
]

print(f"✅ Zufällige Auswahl getroffen: {len(test_queries)} Test-Queries bereit.")

def run_evaluation_for_method(table_name):
    print(f"\n--- 🚀 Starte Evaluierung für: {table_name} ---")
    results_data = []

    for query in test_queries:
        question = query["question"]
        
        contexts = retrieve_from_db(question, table_name, top_k=3)
        
        prompt = f"Beantworte die Frage nur basierend auf diesem Kontext:\n\n{contexts}\n\nFrage: {question}"
        
        try:
            response = azure_model.invoke(prompt)

        except Exception as e:
            print(f"Fehler bei der Anfrage an Azure: {e}")
            continue

        answer = response.content
        
        results_data.append({
            "question": question,
            "answer": answer,
            "contexts": contexts,
            "ground_truth": query["ground_truth"]
        })

    dataset = Dataset.from_list(results_data)
    
    # RAGAS SCORING
    result = evaluate(
        dataset,
        metrics=[
            # Retrieval Evaluierung (Findet das System das Richtige?)
            context_precision, 
            context_recall,
            context_entity_recall,
            
            # Generation Evaluierung (Macht das LLM was Sinnvolles daraus?)
            faithfulness, 
            answer_relevancy, 
            
            # End-to-End Evaluierung (Ist das Endergebnis richtig?)
            answer_correctness,
            answer_similarity,
        ],
        llm=ragas_llm,
        embeddings=ragas_emb
    )
    return result.to_pandas()

final_comparison = {}
all_detailed_results = []

for t in tabellen:
    score_df = run_evaluation_for_method(t)

    score_df["method"] = t
    all_detailed_results.append(score_df)

    final_comparison[t] = score_df.mean(numeric_only=True)

comparison_df = pd.DataFrame(final_comparison).T
print("\n🏆 FINALES RANKING DER CHUNKING-METHODEN:")
print(comparison_df)
comparison_df.to_csv("chunking_evaluation_results.csv")


C:\Users\Martin\AppData\Local\Temp\ipykernel_3516\3905351622.py:5: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\Martin\AppData\Local\Temp\ipykernel_3516\3905351622.py:5: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\Martin\AppData\Local\Temp\ipykernel_3516\3905351622.py:5: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\Martin\AppData\Local\Temp\ipykernel_3

✅ Zufällige Auswahl getroffen: 200 Test-Queries bereit.

--- 🚀 Starte Evaluierung für: chunks_docling_hybrid ---
Fehler bei der Anfrage an Azure: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': True, 'detected': True}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}


Evaluating:   0%|          | 0/1393 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


--- 🚀 Starte Evaluierung für: chunks_docling_hierarchical ---
Fehler bei der Anfrage an Azure: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': True, 'detected': True}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}


...

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


--- 🚀 Starte Evaluierung für: chunks_langchain_recursive ---
Fehler bei der Anfrage an Azure: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': True, 'detected': True}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}
Fehler bei der Anfrage an Azure: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modi

Evaluating:   0%|          | 0/1344 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


🏆 FINALES RANKING DER CHUNKING-METHODEN:
                             context_precision  context_recall  \
chunks_docling_hybrid                 0.933923        0.765494   
chunks_docling_hierarchical           0.927609        0.737856   
chunks_langchain_recursive            0.902778        0.756076   

                             context_entity_recall  faithfulness  \
chunks_docling_hybrid                     0.258948      0.911943   
chunks_docling_hierarchical               0.282808      0.900634   
chunks_langchain_recursive                0.215138      0.889298   

                             answer_relevancy  answer_correctness  \
chunks_docling_hybrid                0.801392            0.617375   
chunks_docling_hierarchical          0.789089            0.637315   
chunks_langchain_recursive           0.786534            0.587917   

                             answer_similarity  
chunks_docling_hybrid                 0.695689  
chunks_docling_hierarchical           0.69288

In [11]:
detailed_df = pd.concat(all_detailed_results, ignore_index=True)
detailed_df.to_csv("chunking_detailed_results.csv", index=False)